In [ ]:
from crewai import Task, Agent, Crew
from crewai_tools import EXASearchTool, ScrapeWebsiteTool
from langchain_ollama import OllamaEmbeddings
from crewai import LLM
import os
import yaml
from crewai import Memory
import re

In [2]:
os.environ["EXA_API_KEY"] = "8836bea6-9c64-4d37-9dc6-7adb97159c15"

#Initialize model through ollama 
#model_id = "ollama/llama3.2:1b"
model_id = "ollama/nemotron-3-nano:4b"
#model_id = "ollama/qwen3.5:9b-mlx"
llm_model = LLM(model=model_id, base_url="http://localhost:11434")

#Initialize local embedder
embedder = OllamaEmbeddings(model = "nomic-embed-text")

# Create the EXASearchTool instance
exa_search_tool = EXASearchTool()

# Create the ScrapeWebsiteTool instance
scrape_website_tool = ScrapeWebsiteTool(embedder = embedder)

/var/folders/nl/s2d1dyx14yl8rtf42tf_0x8m0000gn/T/ipykernel_6618/3667181053.py:13: DeprecationWarning: EXASearchTool is deprecated and will be removed in a future release; use ExaSearchTool instead.
  exa_search_tool = EXASearchTool()


The agents I ll need here are
* Research Planner
* Internet Researcher 
* Fact Checker
* Report Writer

In [3]:
with open('agents1.yaml', 'r') as file:
        agent_config = yaml.safe_load(file)

agent_config

{'research_planner': {'role': 'Research Planner',
  'goal': 'Analyze "{user_query}" and break it down into specific research topics and key questions that need investigation',
  'backstory': 'You are a research strategist who excels at breaking down complex questions into manageable research components. You identify what needs to be researched and create clear research objectives.'},
 'internet_researcher': {'role': 'Internet Researcher',
  'goal': 'Research assigned topics using search and web scraping tools to gather comprehensive, accurate information from multiple credible sources. Ensure all information is properly cited and verified.',
  'backstory': 'You are a skilled researcher with experience in online investigation and data collection. You know how to find reliable sources, extract relevant information, and always verify facts across multiple sources to avoid misinformation or hallucination. You never invent facts and always trace information to its origin.'},
 'fact_checker'

In [4]:
'''
Every agent needs a roal, goal and backstory which ll be taken from the config file
Each key is the individual agent's role, goal and backstory
'''
research_planner = Agent(
        config=agent_config['research_planner'],
        verbose=True,
        max_rpm=150,
        max_iter=15,
        llm = llm_model
        )
internet_researcher = Agent(
        config=agent_config['internet_researcher'],
        tools=[exa_search_tool, scrape_website_tool],
        verbose=True,
        max_rpm=150,
        max_iter=15,
        llm = llm_model
        )
fact_checker = Agent(
        config=agent_config['fact_checker'],
        tools=[exa_search_tool, scrape_website_tool],
        verbose=True,
        max_rpm=150,
        max_iter=15,
        llm = llm_model
        )
report_writer = Agent(
        config=agent_config['report_writer'],
        verbose=True,
        max_rpm=150,
        max_iter=15,
        llm = llm_model
        )

*Guardrails* - https://docs.crewai.com/v1.15.28/en/concepts/tasks#task-guardrails

In [5]:
'''
Receives one parameter and returns a tuple of (Bool, Any)
In case it succeeds, its (True, response)
If it fails its (False, error reason)
'''

# write the custom guardrail function
def write_report_guardrail(output):
    # get the raw output from the TaskOutput object
    try:
        output = output if type(output)==str else output.raw 
    except Exception as e:
        return (False, ("Error retrieving the `raw` argument: "
                        f"\n{str(e)}\n"
                        )
                )
    
    # convert the output to lowercase
    output_lower = output.lower()

    # check that the summary section exists
    if not re.search(r'#+.*summary', output_lower):
        return (False, 
                "The report must include a Summary section with a header like '## Summary'"
                )

    # check that the insights or recommendations sections exist
    if not re.search(r'#+.*insights|#+.*recommendations', output_lower):
        return (False, 
                "The report must include an Insights section with a header like '## Insights'"
                )

    # check that the citations (or references) section exists
    if not re.search(r'#+.*citation', output_lower):
        return (False, 
                "The report must include a Citations section with a header like '## Citations'")

    ### END CODE HERE ###
    return (True, output)

# Like discussed, if none of the checks fail return true,response else the false,reason

In [6]:
test_report_pass = """
# Report title

## Executive Summary
This is a summary.

## Insights
These are the insights.

## Citations
1. Citation 1
2. Citation 2
"""

write_report_guardrail(test_report_pass)

(True,
 '\n# Report title\n\n## Executive Summary\nThis is a summary.\n\n## Insights\nThese are the insights.\n\n## Citations\n1. Citation 1\n2. Citation 2\n')

In [7]:
test_report_fail = """
# Report title

## Executive Summary
This is a summary.
"""

write_report_guardrail(test_report_fail)

(False,
 "The report must include an Insights section with a header like '## Insights'")

In [8]:
with open('tasks1.yaml', 'r') as file:
    task_config = yaml.safe_load(file)

task_config

{'create_research_plan': {'description': 'Break down the research query "{user_query}" into specific topics and key questions that need investigation. Create a focused research plan.',
  'expected_output': 'A research plan with: 1) Main research topics to investigate, 2) Key questions for each topic, 3) Success criteria for the research'},
 'gather_research_data': {'description': 'Using the research plan, search the internet and scrape relevant websites to collect comprehensive information on all identified topics. Verify information across multiple sources and cite all sources used.',
  'expected_output': 'Comprehensive research data including: 1) Information for each research topic with citations, 2) Key facts and statistics, 3) Multiple source perspectives, 4) Source credibility notes'},
 'verify_information_quality': {'description': 'Review all collected research data for accuracy and consistency. Identify any conflicting information, potential misinformation, or gaps that need add

In [9]:
'''
Dictionary with every key as the task
containing the 
description
exepected output
''' 

create_research_plan = Task( 
    config=task_config['create_research_plan'],
    agent=research_planner
)

gather_research_data = Task( 
    config=task_config['gather_research_data'],
    agent=internet_researcher,
)

verify_information_quality = Task( 
    config=task_config['verify_information_quality'],
    agent=fact_checker,
)

write_final_report = Task( 
    config=task_config['write_final_report'],
    agent=report_writer,
    guardrails=[write_report_guardrail], # add the custom guardrail
)

**Hooks** are programmatic logic which ll be executed before and/or after the execution of a crew
https://docs.crewai.com/v1.15.6/en/learn/execution-hooks

In [10]:
'''
After kickoff hook
ll be executed after one crew has finished running 

Contains the output of the crews execution
'''


''' 
This is a very simple kickoff hook 
We are just taking the response, 
checking if there were no errors it should contain the attribute results_output 
And then saving that in a readme/markdown file
'''

def save_file_hook(result):
    try:
        # Get the final report content from the last task output
        if hasattr(result, 'tasks_output') and result.tasks_output:
            report_content = result.tasks_output[-1].raw
        else:
            report_content = str(result)
        
        filename = f"research_report.md"
        
        # Save to file
        with open(filename, 'w', encoding='utf-8') as f:
            f.write(report_content)
        
        print(f"Report successfully saved to: {filename}")
        
    except Exception as e:
        print(f"Error saving report to file: {str(e)}")

In [11]:
#Custom memory for local embeddings
#Crew ai by default uses openai embeddings

local_memory = Memory(
 llm = "ollama/nemotron-3-nano:4b",
 embedder={"provider" : "ollama",
           "config" : {"model_name" : "nomic-embed-text:latest"}}   
)

In [12]:
# So a group of agents and their tasks would constitute a crew ?

deep_research_crew = Crew(
    # include all the agents
    agents=[research_planner, 
            internet_researcher, 
            fact_checker, 
            report_writer],
    # include all the tasks in the order to be executed
    tasks=[create_research_plan, 
           gather_research_data, 
           verify_information_quality, 
           write_final_report],
    
    # add memory to the crew
    memory=local_memory,
    # add the after kickoff hook
    after_kickoff_callbacks=[save_file_hook]
)

In [13]:
input = {
    "user_query" : "Lego Batman 2026 videogame best platform"
}

result = await deep_research_crew.kickoff_async(inputs=input)

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Research Planner                                                                                        │
│                                                                                                                 │
│  Task: Break down the research query "Lego Batman 2026 videogame best platform" into specific topics and key    │
│  questions that need investigation. Create a focused research plan.                                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool search_memory executed with result: Found memories:
- (score=0.86) To determine the best platform to play Lego Batman 2026 videogame, search for platforms that support the game. The recommended platforms include:
  categories: technolog...
[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Research Planner                                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Lego Batman 2026 – Quick‑Reference Summary**                                                                 │
│                                                                                                                 │
│  | Category | Highlights (2023‑2026) |                                                                          │
│  |----------|------------------------|                                                                          │
│  | **Platforms** | • **Primary**: PlayStation 5 (PS5) and Xbox Series X/Series S. <br>• **Secondary**:          │
│  Nintendo Switch (with Switch Online), PC (Steam, GOG etc.). <br>• **Older/limited**: Earlier PS4 (reported     │
│  **lag**, not a target platform). |                                                                             │
│  | **Release** | March 2023 (initial launch) – the **2026** date refers to a rumored new **co‑op mode**. |      │
│  | **Gameplay / Design** | • Faithful building‑and‑battling experience to the original comic spirit. <br>•      │
│  In‑game building (LEGO bricks, plates, minifigures) – can be shared online and used for competitive/creative   │
│  challenges. <br>• Co‑op (rumoured 2026) – team‑up with friends/family. |                                       │
│  | **Graphics / Visuals** | • Rebuilt Batmobile, new detailed environments. <br>• Visual benchmark reports      │
│  (The Verge, 2023) – PS5 shows fastest frame‑rate & load‑times; Xbox SX also strong. |                          │
│  | **Audio** | Score by **Hans Zimmer** with vocals from **Ed Sheeran** (2022). |                               │
│  | **Key Entities** | • Warner Bros. Entertainment (developer). <br>• Hans Zimmer, Ed Sheeran (music). <br>•    │
│  LEGO compatible building system. |                                                                             │
│  | **Critical Reception** | Generally positive – praised for PC flexibility, PS5/ Xbox SX performance, and      │
│  overall game‑play balance. |                                                                                   │
│  | **Industry Notes** | • Marketed as a “bold new chapter” & “true labor of love”. <br>• Player‑test data       │
│  cited (2024) – personal preference vs. objective benchmarks; used to inform platform recommendations. |        │
│  | **Study / Research** | • “Study of platform bias vs. objectivity” – performance scores vs. user‑experience   │
│  comments. <br>• Research on scalability of co‑op features for future builds. |                                 │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### How the Information Fits the Requested Topics                                                              │
│                                                                                                                 │
│  | Topic | Mentioned in the Data |                                                                              │
│  |-------|------------------------|                                                                             │
│  | **Batman** | Game premise, characters (Batman, Riddl

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Internet Researcher                                                                                     │
│                                                                                                                 │
│  Task: Using the research plan, search the internet and scrape relevant websites to collect comprehensive       │
│  information on all identified topics. Verify information across multiple sources and cite all sources used.    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool exa_search_tool executed with result: Error executing tool: Request failed with status code 400: {"requestId":"d3406fa10dcbc0afe41cf47513f31df4","error":"Domain must include a top-level domain: com","tag":"INVALID_DOMAIN"}...
Tool exa_search_tool executed with result: 
Search Time: 1811.5ms
CostDollars: total=0...
Tool exa_search_tool executed with result: 
Search Time: 1221.3ms
CostDollars: total=0...
Tool exa_search_tool executed with result: 
Search Time: 1493ms
CostDollars: total=0...
Tool read_website_content executed with result: Error executing tool: Tool 'Read website content' arguments validation failed: 1 validation error for ScrapeWebsiteToolSchema
website_url
  Field required [type=missing, input_value={'url': 'https://w...
[Finalize] todos_count=0, todos_with_results=0


...

Guardrail 0 blocked (attempt 3/4), retrying due to: The report must include a Summary section with a header like '## Summary'



╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Report Writer                                                                                           │
│                                                                                                                 │
│  Task: Create a comprehensive report that answers the original query "Lego Batman 2026 videogame best           │
│  platform" using all verified research data. Structure it with clear sections, include citations, and provide   │
│  actionable insights.                                                                                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Report Writer                                                                                           │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ## Executive Summary                                                                                           │
│  Based on verified research, the **PC (Xbox Series X|S) version of *Lego Batman 2026* delivers the best         │
│  overall user experience** because of its superior flexibility, customizable settings, and integrated           │
│  cross‑play with console and PC platforms. The **PlayStation 5 excels in raw performance and graphical          │
│  fidelity**, but the PC’s customization and accessibility remain the primary drivers of a “best experience”     │
│  for most players. The recommendation is therefore **PC for user‑experience‑centric play, PS5 for               │
│  performance‑focused play**.                                                                                    │
│                                                                                                                 │
│  ## Summary                                                                                                     │
│  The PC (Xbox Series X|S) version is the most positively reviewed for its user‑driven flexibility and           │
│  customization, making it the optimal choice for a user‑experience‑centric definition of “best platform.”       │
│  However, for those prioritizing peak console performance and visual fidelity, the PlayStation 5 remains the    │
│  top performer. Cross‑play enables both platforms to coexist, allowing shared sessions but not altering the     │
│  core recommendation.                                                                                           │
│                                                                                                                 │
│  ## Detailed Findings                                                                                           │
│  The question “Lego Batman 2026 videogame best platform” is answered as follows:                                │
│                                                                                                                 │
│  1. **User experience & customization** – the PC (Xbox Series X|S) edition receives the highest praise for      │
│  adjustable settings and UI tweaks, directly impacting the player’s immersion and control.                      │
│  2. **Raw performance & visual fidelity** – the PlayStation 5 provides the strongest frame‑rate and load‑time   │
│  consistency, delivering the highest‑quality rendered frames.                                                   │
│  3. **Cross‑play integration** – all three supported platforms (PS5, Xbox Series X|S, PC) can play together     │
│  online, removing platform‑exclusive limitations.                                                               │
│                                                                                                                 │
│  **Overall recommendation:**                                                                                    │
│  - *If your priority is a personalized, configurable experience* → **PC (Xbox Series X|S)**.                    │
│  - *If your priority is the highest performance and visual output* → **PlayStation 5**.                         │
│                                                                                                                 │
│  ## Supporting Evidence and Analysis                   

Report successfully saved to: research_report.md
